In [65]:
%matplotlib inline
from usolver import WGFiber
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np
from scipy.special import jv, kv

In [66]:
class LPModes(WGFiber):
    def __init__(self, v, dia_core):
        super().__init__(v)
        self.v = v  # V number
        self.a = dia_core / 2  # radius of core in micron
        self.mesh_size = 300
        self.xi = np.linspace(-1.5 * self.a, 1.5 * self.a, self.mesh_size)

    def field_core_eq(self, l):
        def wrapper(u, r, phi):
            return jv(l, u * r / self.a) * np.cos(l * phi)
        return wrapper

    def field_clad_eq(self, l):
        def wrapper(u, r, phi):
            w = np.sqrt(self.v * self.v - u * u)
            return (jv(l, u) / kv(l, w)) * kv(l, w * r / self.a) * np.cos(l * phi)
        return wrapper

    def mask_core(self, x, y):
        return np.sqrt(x * x + y * y) <= self.a

    def mask_clad(self, x, y):
        return np.sqrt(x * x + y * y) > self.a

    def LP(self, l, m):
        x, y = np.meshgrid(self.xi, self.xi, indexing="xy")

        u = self.u_lm(l, m)  # Method of 'WGFiber'
        r = np.sqrt(x * x + y * y)
        phi = np.arctan2(x, y)

        fcore = self.field_core_eq(l)
        mfcore = self.mask_core(x, y)  # returns core region 1, clad region 0.
        e_field_core = fcore(u, r, phi) * mfcore

        fclad = self.field_clad_eq(l)
        mfclad = self.mask_clad(x, y)  # returns core region 0, clad region 1.
        e_field_clad = fclad(u, r, phi) * mfclad

        mode_field = e_field_core + e_field_clad
        return mode_field

In [67]:
lps = LPModes(v=4, dia_core=10)

In [68]:
LP01 = lps.LP(0, 1)
LP11 = lps.LP(1, 1)
print(lps.xi)

[-7.5        -7.44983278 -7.39966555 -7.34949833 -7.2993311  -7.24916388
 -7.19899666 -7.14882943 -7.09866221 -7.04849498 -6.99832776 -6.94816054
 -6.89799331 -6.84782609 -6.79765886 -6.74749164 -6.69732441 -6.64715719
 -6.59698997 -6.54682274 -6.49665552 -6.44648829 -6.39632107 -6.34615385
 -6.29598662 -6.2458194  -6.19565217 -6.14548495 -6.09531773 -6.0451505
 -5.99498328 -5.94481605 -5.89464883 -5.84448161 -5.79431438 -5.74414716
 -5.69397993 -5.64381271 -5.59364548 -5.54347826 -5.49331104 -5.44314381
 -5.39297659 -5.34280936 -5.29264214 -5.24247492 -5.19230769 -5.14214047
 -5.09197324 -5.04180602 -4.9916388  -4.94147157 -4.89130435 -4.84113712
 -4.7909699  -4.74080268 -4.69063545 -4.64046823 -4.590301   -4.54013378
 -4.48996656 -4.43979933 -4.38963211 -4.33946488 -4.28929766 -4.23913043
 -4.18896321 -4.13879599 -4.08862876 -4.03846154 -3.98829431 -3.93812709
 -3.88795987 -3.83779264 -3.78762542 -3.73745819 -3.68729097 -3.63712375
 -3.58695652 -3.5367893  -3.48662207 -3.43645485 -3.

In [69]:
I_LP01 = sum(sum(abs(LP01*LP01)))
I_LP11 = sum(sum(abs(LP11*LP11)))

In [70]:
print(I_LP01)
print(I_LP11)

13622.178266386922
4418.564558326013


In [72]:

fig, ax = plt.subplots()
ax.set_aspect('equal')
im = ax.pcolormesh(lps.xi, lps.xi, LP11*LP11, cmap=cm.jet)